# Capstone: End-to-End Alignment Pipeline

This notebook assembles the full alignment pipeline: data curation, instruction tuning, DPO alignment, and evaluation. Each component has been covered individually; here we show how they compose into a coherent training workflow.

## Pipeline Overview

A production alignment pipeline has four stages:

1. **Data Curation**: collect and filter instruction-following examples for SFT; collect preference pairs for DPO
2. **Supervised Fine-Tuning (SFT)**: train on instruction-response pairs to establish format and basic behavior
3. **DPO Alignment**: train on preference pairs to shift behavior toward preferred responses
4. **Evaluation**: measure helpfulness, harmlessness, and honesty; check for capability regression

This order matters: DPO requires a model that already follows instructions reasonably well (the SFT model as reference). Applying DPO to a base model without SFT produces poor results.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable
import random
import math

@dataclass
class SFTExample:
    prompt: str
    response: str
    quality_score: float = 1.0

@dataclass
class DPOPair:
    prompt: str
    chosen: str
    rejected: str
    score_gap: float

@dataclass
class PipelineConfig:
    sft_epochs: int = 2
    dpo_epochs: int = 1
    dpo_beta: float = 0.1
    lora_rank: int = 16
    min_quality_score: float = 0.7
    min_dpo_gap: float = 0.5

def curate_sft_data(raw_examples: list, min_quality: float = 0.7) -> list:
    filtered = [ex for ex in raw_examples if ex.quality_score >= min_quality]
    # Deduplicate by first 50 chars of prompt
    seen = set()
    deduped = []
    for ex in filtered:
        key = ex.prompt[:50]
        if key not in seen:
            seen.add(key)
            deduped.append(ex)
    return deduped

def curate_dpo_data(raw_pairs: list, min_gap: float = 0.5) -> list:
    # Filter pairs where chosen and rejected are too similar
    return [p for p in raw_pairs if p.score_gap >= min_gap]

raw_sft = [
    SFTExample('What is gravity?', 'Gravity is the force of attraction between masses.', 0.9),
    SFTExample('Explain DNA', 'DNA carries genetic information as base pair sequences.', 0.85),
    SFTExample('Write a poem', 'Ok here is a poem.', 0.3),  # low quality
    SFTExample('What is gravity?', 'It is a force.', 0.6),  # duplicate + low quality
    SFTExample('How does HTTPS work?', 'HTTPS uses TLS to encrypt HTTP traffic between client and server.', 0.88),
]

raw_dpo = [
    DPOPair('What is gravity?', 'Gravity is the mutual attraction between masses, described by F=Gm1m2/r2.', 'Gravity makes things fall.', 0.8),
    DPOPair('Explain DNA', 'DNA is a double helix encoding genetic info in A-T and G-C base pairs.', 'DNA is important for life.', 0.6),
    DPOPair('What is 2+2?', 'It is 4.', 'It is 4, definitely.', 0.1),  # too similar
]

sft_data = curate_sft_data(raw_sft, min_quality=0.7)
dpo_data = curate_dpo_data(raw_dpo, min_gap=0.5)

print(f'Data curation results:')
print(f'  SFT: {len(raw_sft)} raw -> {len(sft_data)} curated')
print(f'  DPO: {len(raw_dpo)} raw -> {len(dpo_data)} curated')

In [ ]:
class AlignmentPipeline:
    def __init__(self, config: PipelineConfig, seed: int = 42):
        self.config = config
        self.rng = random.Random(seed)
        self.sft_quality = 0.5   # proxy for model quality
        self.alignment_score = 0.5
        self.capability_retention = 1.0
        self.training_log = []

    def run_sft(self, sft_data: list) -> dict:
        n = len(sft_data)
        avg_quality = sum(ex.quality_score for ex in sft_data) / n if n else 0
        for epoch in range(self.config.sft_epochs):
            gain = avg_quality * 0.1 * (1 - self.sft_quality) + self.rng.gauss(0, 0.01)
            self.sft_quality = min(1.0, self.sft_quality + gain)
            # Small capability cost from specialization
            self.capability_retention -= 0.01 * self.config.lora_rank / 64
        self.training_log.append({
            'stage': 'SFT',
            'examples': n,
            'epochs': self.config.sft_epochs,
            'quality_after': round(self.sft_quality, 3),
            'capability_retention': round(self.capability_retention, 3),
        })
        return self.training_log[-1]

    def run_dpo(self, dpo_data: list) -> dict:
        n = len(dpo_data)
        avg_gap = sum(p.score_gap for p in dpo_data) / n if n else 0
        for epoch in range(self.config.dpo_epochs):
            alignment_gain = avg_gap * 0.15 * self.config.dpo_beta * 10 + self.rng.gauss(0, 0.01)
            self.alignment_score = min(1.0, self.alignment_score + alignment_gain)
        self.training_log.append({
            'stage': 'DPO',
            'pairs': n,
            'epochs': self.config.dpo_epochs,
            'beta': self.config.dpo_beta,
            'alignment_after': round(self.alignment_score, 3),
        })
        return self.training_log[-1]

    def evaluate(self, eval_fn: Callable) -> dict:
        scores = eval_fn(self.sft_quality, self.alignment_score, self.capability_retention)
        self.training_log.append({'stage': 'eval', **scores})
        return scores

def mock_eval(sft_quality, alignment, capabilities):
    return {
        'helpfulness': round(sft_quality * 0.9, 3),
        'harmlessness': round(alignment * 0.85 + 0.1, 3),
        'capability_retention': round(capabilities, 3),
        'overall': round((sft_quality + alignment + capabilities) / 3, 3),
    }

config = PipelineConfig(sft_epochs=2, dpo_epochs=1, dpo_beta=0.1, lora_rank=16)
pipeline = AlignmentPipeline(config)

print('Running alignment pipeline...')
print('\n1. Data curation')
print(f'   SFT: {len(sft_data)} examples, DPO: {len(dpo_data)} pairs')
print('\n2. SFT stage')
sft_result = pipeline.run_sft(sft_data)
for k, v in sft_result.items():
    print(f'   {k}: {v}')
print('\n3. DPO stage')
dpo_result = pipeline.run_dpo(dpo_data)
for k, v in dpo_result.items():
    print(f'   {k}: {v}')
print('\n4. Evaluation')
eval_result = pipeline.evaluate(mock_eval)
for k, v in eval_result.items():
    print(f'   {k}: {v}')

## Series 12 Recap: The Fine-Tuning and Alignment Stack

Across these 12 notebooks we built the complete picture:

1. **Fine-tuning landscape**: why full fine-tuning stopped scaling at 7B+
2. **LoRA**: low-rank decomposition as the PEFT standard
3. **QLoRA**: 4-bit quantization enabling large-model fine-tuning on single GPUs
4. **Instruction tuning**: data quality over quantity; format and diversity
5. **RLHF reward modeling**: Bradley-Terry preferences; annotation quality challenges
6. **RLHF PPO**: the RL loop, KL penalty, instability history
7. **DPO**: eliminating the RL loop with a closed-form preference objective
8. **ORPO & SimPO**: reference-free alignment and length normalization
9. **Constitutional AI**: self-critique, explicit principles, and RLAIF
10. **Alignment evaluation**: measuring HHH, tracking over-refusal
11. **Catastrophic forgetting**: LoRA merging and continual learning
12. **Capstone**: end-to-end pipeline from data curation to aligned model

The through-line: every alignment method is a tradeoff between helpfulness, safety, training cost, and capability retention. There is no single correct choice — the right pipeline depends on your use case, data budget, and hardware constraints.